In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
import pandas as pd
import csv

# We use quoting=csv.QUOTE_NONE (or 3) to prevent pandas from expecting closing quotes
# inside text reviews, which fixes the EOF (End Of File) parsing error.
data = pd.read_csv("/content/IMDB Dataset.csv", quoting=csv.QUOTE_NONE, on_bad_lines='skip')

data

,,,,,,,,,,,,,,,,,,,,,,,,,,review,sentiment
"""One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right",as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence,which set in right from the word GO. Trust me,this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs,sex or violence. Its is hardcore,in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City,an experimental section of the prison where all the cells have glass fronts and face inwards,so privacy is not high on the agenda. Em City is home to many..Aryans,Muslims,gangstas,Latinos,Christians,Italians,Irish and more....so scuffles,death stares,dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences,forget charm,forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal,I couldn't say I was ready for it,but as I watched more,I developed a taste for Oz,and got accustomed to the high levels of graphic violence. Not just violence,but injustice (crooked guards who'll be sold out for a nickel,inmates who'll kill on order and get away with it,well mannered,middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz,you may become comfortable with what is uncom...,positive
"""A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting",and sometimes discomforting,"sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only """"has got all the polari"""" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries",not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which,rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses,"particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"""I thought this was a wonderful way to spend time on a too hot summer weekend",sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic,but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction,I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.<br /><br />This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson,"in this she managed to tone down her """"sexy"""" image and jumped right into a average",but spirited young woman.<br /><br />This may not be the crown jewel of his career,"but it was wittier than """"Devil Wears Prada"""" and more interesting than """"Superman"""" a great comedy to go see with friends.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"""Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<b

In [6]:
data = data.dropna()

In [7]:
texts = data["review"].astype(str)
texts

,,,,,,,,,,,,,,,,,,,,,,,,,,review
"""One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right",as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence,which set in right from the word GO. Trust me,this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs,sex or violence. Its is hardcore,in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City,an experimental section of the prison where all the cells have glass fronts and face inwards,so privacy is not high on the agenda. Em City is home to many..Aryans,Muslims,gangstas,Latinos,Christians,Italians,Irish and more....so scuffles,death stares,dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences,forget charm,forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal,I couldn't say I was ready for it,but as I watched more,I developed a taste for Oz,and got accustomed to the high levels of graphic violence. Not just violence,but injustice (crooked guards who'll be sold out for a nickel,inmates who'll kill on order and get away with it,well mannered,middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz,you may become comfortable with what is uncom...
"""Gamers: DR is not a fancy made movie",it's more like amateur video. Horrible magic effect,really fake fireball,terribly made dungeon,castle,village...... sword,axe,shield,robe,plate..... okok... everything. You will need about 10 minutes to adjust your expecting on visual,then you will get 105 minutes of fun.<br /><br />I'm from Hongkong and it's really hard to find RPers,none of my friends play RPG and I always fancy to be one of the character in the world of D&D. Watching Gamers: DR just show me what would it be like to be a gamer. You see rule books,dices,game set,"etc etc etc; You hear terms like """"fighter""""","""""wizard""""","""""hit point""""","""""level""""","""""character""""","""""flaming hand""""","""""Chaotic Evil"""".<br /><br />What RPG fancy me is that it let you do anything u want to",not bonded by software RPG. Gamers: DR provide the same element,you wont know what happen next and it probably just make you laugh to dead. The movie goes both gamers's real life as well as in the D&D world. You will hear the gamer cast the dice when the character in game take action,which make you feel you really participle in the game.<br /><br />I don't want to spoil anything,but in short,Gamers: DR is a must watch movie for RPG lovers. For people never play RPG game,"I'm sure you still get many fun from it."""
"""First",let me just comment on what I liked about the movie. The special effects were fantastic,and very rarely did I feel like I was watching a video game. There,that is the last nice thing I have to say about this film. In fact,I would just like everyone reading this to take note that I can't even put into words how hard it was for me to write this review without swearing. <br /><br />I have innumerable complaints about the film,"but four major complaints jump to mind. My first major complaint has to do with the incredible cheesiness of the """"plot twist"""" (if you can call it that since most people probably saw it coming a mile away) where Lois's 5 year-old son turns out to be the super-powered child of Superman. When the crying super-child throws a piano at Lex's henchman to save his mother",I almost got up and left the theater. Singer could have made a much better Superman movie without resorting to cheap gimmicks like a seemingly frag

In [8]:
Sentiment = data["sentiment"]
Sentiment

,,,,,,,,,,,,,,,,,,,,,,,,,,sentiment
"""One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right",as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence,which set in right from the word GO. Trust me,this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs,sex or violence. Its is hardcore,in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City,an experimental section of the prison where all the cells have glass fronts and face inwards,so privacy is not high on the agenda. Em City is home to many..Aryans,Muslims,gangstas,Latinos,Christians,Italians,Irish and more....so scuffles,death stares,dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences,forget charm,forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal,I couldn't say I was ready for it,but as I watched more,I developed a taste for Oz,and got accustomed to the high levels of graphic violence. Not just violence,but injustice (crooked guards who'll be sold out for a nickel,inmates who'll kill on order and get away with it,well mannered,middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz,positive
"""Gamers: DR is not a fancy made movie",it's more like amateur video. Horrible magic effect,really fake fireball,terribly made dungeon,castle,village...... sword,axe,shield,robe,plate..... okok... everything. You will need about 10 minutes to adjust your expecting on visual,then you will get 105 minutes of fun.<br /><br />I'm from Hongkong and it's really hard to find RPers,none of my friends play RPG and I always fancy to be one of the character in the world of D&D. Watching Gamers: DR just show me what would it be like to be a gamer. You see rule books,dices,game set,"etc etc etc; You hear terms like """"fighter""""","""""wizard""""","""""hit point""""","""""level""""","""""character""""","""""flaming hand""""","""""Chaotic Evil"""".<br /><br />What RPG fancy me is that it let you do anything u want to",not bonded by software RPG. Gamers: DR provide the same element,you wont know what happen next and it probably just make you laugh to dead. The movie goes both gamers's real life as well as in the D&D world. You will hear the gamer cast the dice when the character in game take action,which make you feel you really participle in the game.<br /><br />I don't want to spoil anything,but in short,Gamers: DR is a must watch movie for RPG lovers. For people never play RPG game,positive
"""First",let me just comment on what I liked about the movie. The special effects were fantastic,and very rarely did I feel like I was watching a video game. There,that is the last nice thing I have to say about this film. In fact,I would just like everyone reading this to take note that I can't even put into words how hard it was for me to write this review without swearing. <br /><br />I have innumerable complaints about the film,"but four major complaints jump to mind. My first major complaint has to do with the incredible cheesiness of the """"plot twist"""" (if you can call it that since most people probably saw it coming a mile away) where Lois's 5 year-old son turns out to be the super-powered child of Superman. When the crying super-child throws a piano at Lex's henchman to save his mother",I almost got up and left the theater. Singer could have made a much better Superman movie without resorting to cheap gimmicks like a seemingly fragile but latently super-powered illegitimate child. It's been 5 days since

In [10]:
encoder = LabelEncoder()
labels = encoder.fit_transform(Sentiment)

print("Encoded labels:", labels)
print("Label mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

Encoded labels: [1 1 0 0 0 0 0 1 0 0 1 0 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 0 0 1 0 0 0 1 1 1 0
 0 0 0 0 0 1 1 1 1 0 0 0 1 1 1 0 0 0 0 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 1 1 1
 0 0 0 1 0 0 1 0 1 1 1 1 1 0 0 0 0 0 1 0 0 0 1 1 0 1 0 1 1 1 0 1 0 0 1 0 0
 0 0 1 1 1 0 1 0 1 0 1 1 1 1 1 1 0 1 1 1 1 0 1 0 1 0 1 1 1 0 0 0 1 0 0 0 1
 0 1 0 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 1 1 1 1 0 1 1 1 0 0 0 1 1 0 0 0 0 1 1
 0 1 0 1 1 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 1 1 1 1 0 1 1 0 1 1 1 0
 1 0 1 1 0 1 1 0 1 1 1 1 1 1 1 1 0 1 0 1 0 1 0 0 1 1 1 1 1 1 0 0 0 1 1 0 1
 0 0 0 1 1 0 0 0 0 0 1 1 1 1 0 0 1 1 0 0 0 0 0 1 1 0 0 1 1 1 0 1 0 1 0 0 1
 1 0 0 0 1 1 0 1 0 1 1 0 0 1 1 1 0 0 1 0 1 1 0 1 1 1 0 1 0 1 0 1 0 1 0 0 1
 1 0 0 1 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 1 1 0 1 1]
Label mapping: {'negative': np.int64(0), 'positive': np.int64(1)}


In [11]:
tokenizer = Tokenizer(
    lower=True,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(texts)

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Re-initialize and fit tokenizer since previous cell failed
tokenizer = Tokenizer(lower=True, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
sequences

[[20,
  100,
  243,
  594,
  28,
  42,
  9,
  595,
  244,
  245,
  45,
  20,
  39,
  61,
  11,
  596,
  28,
  37,
  597,
  187],
 [87, 131, 20, 101, 61, 74, 344, 29, 8],
 [25,
  598,
  102,
  5,
  34,
  345,
  599,
  20,
  11,
  2,
  346,
  5,
  103,
  61,
  2,
  115,
  246,
  132,
  347,
  7,
  150,
  104,
  247,
  4,
  37,
  46],
 [347, 600, 248, 7, 2, 88],
 [2,
  601,
  37,
  348,
  4,
  602,
  249,
  349,
  6,
  6,
  603,
  604,
  9,
  605,
  133,
  7,
  350,
  57,
  2,
  134,
  606,
  4,
  607,
  6,
  6,
  89,
  46,
  608,
  105,
  609,
  40,
  249,
  349,
  9,
  3,
  116,
  46,
  7,
  610,
  21,
  611,
  612,
  37,
  351,
  6,
  6,
  613,
  614,
  26,
  35,
  615,
  616,
  617,
  14,
  618,
  31,
  2,
  115,
  619,
  6,
  6,
  250,
  620,
  5,
  621,
  622,
  26,
  623,
  352,
  40,
  2,
  151,
  624],
 [5, 20, 34, 152, 3, 625],
 [43,
  26,
  62,
  38,
  75,
  5,
  626,
  188,
  627,
  628,
  2,
  353,
  5,
  629,
  4,
  35,
  630,
  63,
  16,
  32,
  189,
  12,
  106],
 [33, 188

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)

X

array([[ 20, 100, 243, ...,   0,   0,   0],
       [ 87, 131,  20, ...,   0,   0,   0],
       [ 25, 598, 102, ...,   0,   0,   0],
       ...,
       [ 96, 572,   2, ...,   0,   0,   0],
       [ 55,  41, 100, ...,   0,   0,   0],
       [ 56,   9,  23, ...,   0,   0,   0]], dtype=int32)

In [14]:
y = np.array(labels)
y

array([1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1,
       1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1,
       0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1,

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 1881


In [17]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

model.add(
    Bidirectional(
        LSTM(
            128,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        len(set(labels)),
        activation='softmax'
    )
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

In [20]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.4667 - loss: 0.6953 - val_accuracy: 0.5278 - val_loss: 0.6912
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.5439 - loss: 0.6893 - val_accuracy: 0.5278 - val_loss: 0.6911
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5368 - loss: 0.6915 - val_accuracy: 0.5278 - val_loss: 0.6903
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.5333 - loss: 0.6808 - val_accuracy: 0.5278 - val_loss: 0.6892


In [21]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nAccuracy :", accuracy)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 242ms/step - accuracy: 0.5278 - loss: 0.6912

Accuracy : 0.5277777910232544


In [25]:
def detect_sentiment(sentence):
    sentence = sentence.lower()
    seq = tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # The model outputs a probability distribution over classes because of activation='softmax'
    prediction = model.predict(seq, verbose=0)[0]

    # Find the index with the highest probability
    predicted_index = np.argmax(prediction)
    confidence = prediction[predicted_index] * 100

    sentiment = encoder.inverse_transform([predicted_index])[0]

    return sentiment, confidence

In [26]:
review = "This movie was amazing and the acting was excellent!"

sentiment, confidence = detect_sentiment(review)

print("Sentiment:", sentiment)
print(f"Confidence: {confidence:.2f}%")

Sentiment: positive
Confidence: 52.93%
